# Lab 05.3 — A Simple KerasTuner Example
Tune the width of two hidden layers and the learning rate on Fashion MNIST.

The original saved error was `KeyboardInterrupt`: training was interrupted, rather than a recorded model or import exception. The final evaluation also referred to undefined `x_test` and `y_test`. This version separates training, validation, and test data and uses the current `keras_tuner` import.

## 1. Install and import
If packages are missing, uncomment the installation line, run it, then restart the kernel and continue from the imports.

In [ ]:
# %pip install --upgrade keras-tuner tensorflow scikit-learn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt
from sklearn.model_selection import train_test_split

keras.utils.set_random_seed(7)
print('TensorFlow:', tf.__version__)
print('KerasTuner:', kt.__version__)


## 2. Prepare the data
Use 10,000 training images and 2,000 validation images for a quicker teaching example. They are separate subsets of the original training set. The original test set is reserved for the final evaluation.

Pixel values are converted from 0–255 to 0–1. Increase the sample counts later for a larger experiment.

In [ ]:
(x, y), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
x_train, x_val, y_train, y_val = train_test_split(
    x, y, train_size=10000, test_size=2000,
    stratify=y, random_state=7
)
x_train = x_train.astype('float32') / 255.0
x_val = x_val.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
print('Training:', x_train.shape)
print('Validation:', x_val.shape)
print('Test:', x_test.shape)


## 3. Define tunable parameters
- **`units1`, `units2`:** width of each hidden layer. More units increase capacity and computation, but may also increase overfitting.
- **`learning_rate`:** size of Adam's updates. Too small can train slowly; too large can make training unstable.

Each width has 8 choices (32–256, step 32), and the learning rate has 3 choices: 192 possible combinations. Random Search tests only 5 of them. Batch size and epochs are fixed in this example.

In [ ]:
def build_model(hp):
    units1 = hp.Int('units1', min_value=32, max_value=256, step=32)
    units2 = hp.Int('units2', min_value=32, max_value=256, step=32)
    learning_rate = hp.Choice('learning_rate', values=[0.01, 0.001, 0.0001])

    model = keras.Sequential([
        keras.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(units1, activation='relu'),
        layers.Dense(units2, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


## 4. Configure Random Search
`max_trials=5` tests up to five parameter combinations. `executions_per_trial=1` trains each once; the original value of 3 required 15 training runs.

`overwrite=True` starts a fresh search and replaces earlier tuner results **in this example's project folder**. Use `overwrite=False` to resume the same unchanged experiment. Use a different `project_name` to keep a separate experiment.

In [ ]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    executions_per_trial=1,
    seed=7,
    overwrite=True,
    directory='keras_tuner_results',
    project_name='lab05_3_fashion_mnist'
)
tuner.search_space_summary()


## 5. Run the search
Each model trains for at most five epochs, using batches of 64. Early stopping can end an unpromising training run when validation accuracy stops improving for two epochs. Epoch-level output shows progress.

If you stop this cell, Python may report `KeyboardInterrupt`. Let the search finish before running the result cells.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_accuracy', mode='max', patience=2
)
tuner.search(
    x_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(x_val, y_val),
    callbacks=[early_stop],
    verbose=2
)


## 6. Inspect the best parameters
The selected settings are the best among the tested trials using validation accuracy; they are not guaranteed to be the best across the full search space.

In [ ]:
tuner.results_summary()
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print('First hidden layer:', best_hp.get('units1'))
print('Second hidden layer:', best_hp.get('units2'))
print('Learning rate:', best_hp.get('learning_rate'))


## 7. Evaluate on the separate test set
Load the best validation checkpoint and evaluate it once. Do not select hyperparameters using test accuracy. An optimizer-state loading warning, if shown, is different from an exception; the exact message is needed to diagnose it.

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]
test_loss, test_acc = best_model.evaluate(x_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.3f}')


### Documentation
[KerasTuner getting started](https://keras.io/keras_tuner/getting_started/) · [RandomSearch API](https://keras.io/keras_tuner/api/tuners/random/)